In [0]:
from pyspark.sql import functions as F
start_date='2024-01-01'
end_date='2024-12-31'

df=(spark.sql(f"""
             select explode(
                 sequence(
                     to_date("{start_date}"),
                     to_date("{end_date}"),
                     interval 1 month )) as month_start_date
""")
)

df=(df.withColumn("date_key",F.date_format(F.col("month_start_date"),"yyyyMM").cast("int"))
                 .withColumn("month_name",F.date_format(F.col("month_start_date"),"MMMM"))
                 .withColumn("year",F.date_format(F.col("month_start_date"),"yyyy"))
                 .withColumn("month_short_name",F.date_format(F.col("month_start_date"),"MMM"))
                 .withColumn("quarter",F.concat(F.lit("Q-"),F.quarter(F.col("month_start_date"))))
                 .withColumn("year_Quater",F.concat(F.col("year"),F.lit("-Q-"),F.quarter(F.col("month_start_date"))))
)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", True) \
    .saveAsTable("fmcg.gold.dim_date")
